# scikit-learn Recipes — Exercises

Companion to the note [scikit-learn Recipes](scikit-learn%20Recipes.md).

Practise the note's recipes on scikit-learn's bundled toy datasets (no downloads): stratified splits, pipelines that avoid leakage, cross-validation, `GridSearchCV`, `ColumnTransformer` for mixed data, custom transformers and estimators, metrics (classification report, ROC AUC, confusion matrix), threshold tuning, and the unsupervised PCA + k-means recipe.

**17 exercises** · 🧠 Concept ×4 · ✍️ By hand ×4 · 💻 Code ×9

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin, clone
from sklearn.datasets import load_iris, load_breast_cancer, load_wine
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import (train_test_split, cross_val_score, cross_val_predict, GridSearchCV,
                                     StratifiedKFold)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import NearestCentroid
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, f1_score, accuracy_score,
                             roc_auc_score, adjusted_rand_score, classification_report)
rng = np.random.default_rng(0)
X_bc, y_bc = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.2, stratify=y_bc, random_state=0)
cv5 = StratifiedKFold(5, shuffle=True, random_state=0)

## Part A · Concepts

### Exercise 1 · fit, transform, predict
*🧠 Concept · ★☆☆*

Explain the difference between `fit`, `transform`, `fit_transform`, `predict` and `score`. Which of them may be called on the
test set? What goes wrong in this code?

```python
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.fit_transform(X_te)
```

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

`fit` learns parameters from data (means, weights, ...); everything else *uses* learned parameters.

</details>

<details>
<summary><b>✅ Solution</b></summary>

`fit` learns parameters (a scaler's mean/std, a model's weights); `transform` applies a fitted transformer; `fit_transform` does
both on the same data (sometimes more efficiently); `predict` applies a fitted model; `score` predicts and evaluates (accuracy
for classifiers, $R^2$ for regressors). On the test set only `transform`, `predict`, `predict_proba` and `score` are allowed.
The bug: the second line **re-fits** the scaler on the test set, so test data is scaled with different statistics than training
data. It should be `scaler.transform(X_te)`. A `Pipeline` makes this mistake impossible.

</details>

### Exercise 2 · Leakage and pipelines
*🧠 Concept · ★★☆*

Why does the note say "Pipeline avoids leakage (scaler is fit inside each CV fold)"? Describe what happens to the validation fold
in `cross_val_score(make_pipeline(StandardScaler(), SVC()), X, y)` versus scaling `X` once before calling
`cross_val_score(SVC(), X_scaled, y)`. Why does the parameter grid use names like `svc__C`?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

In each CV split, the validation fold must play the role of unseen data for **every** step that learns something.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With the pipeline, each fold calls `pipe.fit(X_train_fold)`, so the scaler sees only the training part, and the validation fold is
transformed with those statistics: an honest simulation of deployment. Scaling all of `X` first lets validation rows influence the
means/stds used in training. For a scaler the effect is small, but for data-dependent steps like feature selection, imputation or
target encoding it can be huge (Exercise 10). `make_pipeline` names steps after their lowercased class (`standardscaler`, `svc`),
and `step__param` addresses the parameter `param` of the step `step`, e.g. `svc__C`.

</details>

### Exercise 3 · Stratification and shuffling
*🧠 Concept · ★☆☆*

What do `stratify=y` in `train_test_split` and `StratifiedKFold(5, shuffle=True, random_state=0)` guarantee? When is stratification
important, and when would shuffling be **wrong**? ([[Cross-Validation and Model Selection]])

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think of a dataset with 5% positives, and of time series.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Stratification keeps the class proportions (approximately) equal in every split, so no fold ends up with very few positives;
this matters for imbalanced data and small datasets, and reduces the variance of CV estimates. Shuffling randomises which samples
land in which fold (with `random_state` for reproducibility); without it, sorted data (e.g. all class 0 first) gives
degenerate folds. Shuffling is wrong for time series (the future would leak into training, use `TimeSeriesSplit`) and when samples
are grouped (several rows per patient: use `GroupKFold`).

</details>

### Exercise 4 · Reading a classification report
*🧠 Concept · ★★☆*

A classifier for a rare disease (1% positives) reports accuracy 0.99 and macro-averaged F1 0.50. What has probably happened?
Explain precision, recall, F1, and the difference between macro and weighted averages in `classification_report`.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

What do accuracy and F1 look like for a model that always predicts 'healthy'?

</details>

<details>
<summary><b>✅ Solution</b></summary>

Most likely the model predicts the majority class for everyone: accuracy $=0.99$, F1 of the healthy class $\approx1$ and of the
disease class $=0$, so the macro average is about $0.5$.
Precision $=\frac{TP}{TP+FP}$ (how many flagged cases are real), recall $=\frac{TP}{TP+FN}$ (how many real cases are found),
F1 is their harmonic mean. **Macro** averages per-class scores equally (sensitive to minority classes), **weighted** averages them
by support (dominated by the majority class). For imbalanced problems look at per-class recall/precision, PR curves or ROC AUC.

</details>

## Part B · By hand

### Exercise 5 · How many fits does GridSearchCV run?
*✍️ By hand · ★☆☆*

The note's grid is `{"svc__C": [0.1, 1, 10, 100], "svc__gamma": [1e-3, 1e-2, 1e-1]}` with `cv=5` and the default
`refit=True`. How many times is the pipeline fitted in total? How many if you used `RandomizedSearchCV(n_iter=5, cv=5)`?

In [ ]:
n_fits_grid = None
n_fits_random = None

from sklearn.model_selection import ParameterGrid
check('grid search fits', n_fits_grid, len(ParameterGrid({"svc__C": [0.1, 1, 10, 100], "svc__gamma": [1e-3, 1e-2, 1e-1]})) * 5 + 1)
check('randomized search fits', n_fits_random, 5 * 5 + 1)

<details>
<summary><b>💡 Hint</b></summary>

Grid size × number of folds, plus the final refit on the whole training set.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$4\times3 = 12$ candidates × 5 folds $= 60$ fits, plus $1$ refit $=61$. Randomised search: $5\times5+1=26$. Grid search cost grows multiplicatively with each hyperparameter; randomised search keeps the budget fixed.

```python
n_fits_grid = 61
n_fits_random = 26
```

</details>

### Exercise 6 · Metrics from a confusion matrix
*✍️ By hand · ★☆☆*

A binary classifier has $TP=40$, $FP=10$, $FN=5$, $TN=45$. Compute accuracy, precision, recall and F1.

In [ ]:
acc_h = None; prec_h = None; rec_h = None; f1_h = None

_yt = np.r_[np.ones(40), np.zeros(10), np.ones(5), np.zeros(45)]
_yp = np.r_[np.ones(40), np.ones(10), np.zeros(5), np.zeros(45)]
check('accuracy', acc_h, accuracy_score(_yt, _yp)); check('precision', prec_h, precision_score(_yt, _yp))
check('recall', rec_h, recall_score(_yt, _yp)); check('F1', f1_h, f1_score(_yt, _yp))

<details>
<summary><b>💡 Hint</b></summary>

$F_1 = \frac{2PR}{P+R} = \frac{2TP}{2TP+FP+FN}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Accuracy $=85/100=0.85$; precision $=40/50=0.8$; recall $=40/45\approx0.889$; $F_1 = 80/95\approx0.842$.

```python
acc_h = 0.85; prec_h = 0.8; rec_h = 40 / 45; f1_h = 80 / 95
```

</details>

### Exercise 7 · ROC AUC as a ranking probability
*✍️ By hand · ★★☆*

ROC AUC equals the probability that a random positive gets a higher score than a random negative. Compute it by counting pairs for
positive scores $(0.9, 0.8, 0.4)$ and negative scores $(0.7, 0.3, 0.2)$.

In [ ]:
auc_hand = None

check('AUC', auc_hand, roc_auc_score([1, 1, 1, 0, 0, 0], [0.9, 0.8, 0.4, 0.7, 0.3, 0.2]))

<details>
<summary><b>💡 Hint</b></summary>

There are $3\times3=9$ (positive, negative) pairs; count those where the positive scores higher.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$0.9$ and $0.8$ beat all three negatives ($6$ pairs); $0.4$ beats $0.3$ and $0.2$ but not $0.7$ ($2$ pairs). AUC $=8/9\approx0.889$. AUC only depends on the *ranking* of scores, not on a threshold or calibration.

```python
auc_hand = 8 / 9
```

</details>

### Exercise 8 · What StandardScaler computes
*✍️ By hand · ★☆☆*

A training column has values $(2, 4, 6, 8)$. What are `scaler.mean_`, `scaler.scale_` (sklearn uses the **population** standard
deviation, `ddof=0`), and the transformed value of a test point $x=8$? And of $x=11$?

In [ ]:
mean_h = None; scale_h = None; z8 = None; z11 = None

_s = StandardScaler().fit(np.array([[2.], [4.], [6.], [8.]]))
check('mean_', mean_h, _s.mean_[0]); check('scale_', scale_h, _s.scale_[0])
check('z(8)', z8, _s.transform([[8.]])[0, 0]); check('z(11)', z11, _s.transform([[11.]])[0, 0])

<details>
<summary><b>💡 Hint</b></summary>

Deviations from the mean are $(-3,-1,1,3)$; the variance is their mean square.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Mean $5$, variance $(9+1+1+9)/4 = 5$, scale $\sqrt5\approx2.236$. $z(8) = 3/\sqrt5\approx1.342$, $z(11) = 6/\sqrt5\approx2.683$. A test value outside the training range just maps outside the training range of $z$.

```python
mean_h = 5.0; scale_h = np.sqrt(5); z8 = 3 / np.sqrt(5); z11 = 6 / np.sqrt(5)
```

</details>

## Part C · Recipes in code

### Exercise 9 · Split, pipeline, cross-validate
*💻 Code · ★☆☆*

Using `X_tr, y_tr` (the stratified split of the breast-cancer data in the setup), build `pipe_lr = make_pipeline(StandardScaler(),
LogisticRegression(max_iter=1000))`, compute 5-fold CV accuracy with `cv5`, store the mean in `cv_mean_lr`, then fit on the
whole training set and store the test accuracy in `test_acc_lr`. Print the classification report for the test set.

In [ ]:
pipe_lr = None
cv_mean_lr = None
test_acc_lr = None

check('stratified split keeps class balance', abs(y_tr.mean() - y_te.mean()) < 0.01, True)
check('CV accuracy > 0.95', None if cv_mean_lr is None else cv_mean_lr > 0.95, True)
check('test accuracy > 0.95', None if test_acc_lr is None else test_acc_lr > 0.95, True)

<details>
<summary><b>💡 Hint</b></summary>

`cross_val_score(pipe_lr, X_tr, y_tr, cv=cv5).mean()`, then `pipe_lr.fit(X_tr, y_tr).score(X_te, y_te)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

CV accuracy (about 0.98) estimates generalisation using only training data; the test set is touched once at the very end. If you tune on the test score, it stops being an unbiased estimate.

```python
pipe_lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
cv_mean_lr = cross_val_score(pipe_lr, X_tr, y_tr, cv=cv5).mean()
test_acc_lr = pipe_lr.fit(X_tr, y_tr).score(X_te, y_te)
print(cv_mean_lr, test_acc_lr)
print(classification_report(y_te, pipe_lr.predict(X_te)))
```

</details>

### Exercise 10 · Demonstrating leakage
*💻 Code · ★★☆*

Make pure-noise data: `X_noise` with 100 samples × 5000 features (standard normal, seed 1) and balanced random labels. Compare:

- **leaky**: select the 20 best features with `SelectKBest(f_classif, k=20)` on *all* data, then `cross_val_score(LogisticRegression(), X_sel, y, cv=cv5)`;
- **honest**: put `SelectKBest` and `LogisticRegression` in a pipeline and cross-validate that.

Store the mean accuracies in `acc_leaky` and `acc_honest`. Which one tells the truth?

In [ ]:
r_noise = np.random.default_rng(1)
X_noise = r_noise.normal(size=(100, 5000)); y_noise = np.repeat([0, 1], 50)
acc_leaky = None
acc_honest = None

check('leaky CV looks great', None if acc_leaky is None else acc_leaky > 0.75, True)
check('honest CV is near chance', None if acc_honest is None else abs(acc_honest - 0.5) < 0.15, True)

<details>
<summary><b>💡 Hint</b></summary>

`X_sel = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)` for the leaky version; `make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())` for the honest one.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The labels are random, so the true accuracy is 50%. The leaky version (about 0.88) picked the 20 features that happen to correlate with the labels *including the validation rows*, so CV is fooled. The pipeline re-selects features inside each fold and reports about 0.5. This is the classic mistake from *The Elements of Statistical Learning* §7.10.2.

```python
r_noise = np.random.default_rng(1)
X_noise = r_noise.normal(size=(100, 5000)); y_noise = np.repeat([0, 1], 50)
X_sel = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
acc_leaky = cross_val_score(LogisticRegression(), X_sel, y_noise, cv=cv5).mean()
acc_honest = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression()), X_noise, y_noise, cv=cv5).mean()
print(acc_leaky, acc_honest)
```

</details>

### Exercise 11 · Hyperparameter search with GridSearchCV
*💻 Code · ★★☆*

Run the note's grid search (`StandardScaler` + `SVC(kernel="rbf")`, grid over `svc__C` and `svc__gamma` as in the note) on
`X_tr, y_tr` with `cv=cv5`. Store the fitted search in `grid`. Inspect `grid.cv_results_` as a DataFrame sorted by
`rank_test_score`, and store the test accuracy of the refitted best model in `grid_test_acc`.

In [ ]:
param_grid = {"svc__C": [0.1, 1, 10, 100], "svc__gamma": [1e-3, 1e-2, 1e-1]}
grid = None
grid_test_acc = None

check('12 candidates evaluated', None if grid is None else len(grid.cv_results_['params']), 12)
check('best params come from the grid', None if grid is None else (grid.best_params_['svc__C'] in param_grid['svc__C'] and grid.best_params_['svc__gamma'] in param_grid['svc__gamma']), True)
check('best CV score = max of mean_test_score', None if grid is None else grid.best_score_, None if grid is None else np.max(grid.cv_results_['mean_test_score']))
check('test accuracy > 0.94', None if grid_test_acc is None else grid_test_acc > 0.94, True)

<details>
<summary><b>💡 Hint</b></summary>

`grid = GridSearchCV(make_pipeline(StandardScaler(), SVC(kernel='rbf')), param_grid, cv=cv5).fit(X_tr, y_tr)`; `pd.DataFrame(grid.cv_results_)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

`best_score_` is the mean CV accuracy of the winner, which is optimistically biased because it was selected as the maximum of 12 noisy estimates; the held-out test score is the honest number. For an unbiased estimate of the whole tuning procedure use nested CV (`cross_val_score(grid, X, y)`).

```python
param_grid = {"svc__C": [0.1, 1, 10, 100], "svc__gamma": [1e-3, 1e-2, 1e-1]}
grid = GridSearchCV(make_pipeline(StandardScaler(), SVC(kernel='rbf')), param_grid, cv=cv5).fit(X_tr, y_tr)
res = pd.DataFrame(grid.cv_results_).sort_values('rank_test_score')
print(res[['param_svc__C', 'param_svc__gamma', 'mean_test_score', 'std_test_score']].head())
grid_test_acc = grid.score(X_te, y_te)
print(grid.best_params_, grid_test_acc)
```

</details>

### Exercise 12 · ColumnTransformer for mixed data
*💻 Code · ★★☆*

Build a mixed-type DataFrame from the wine data: the 13 numeric columns with 5% of entries set to `NaN`, plus a categorical column
`"region"` (values `"north"`, `"south"`, `"east"`, randomly assigned). Create a `ColumnTransformer` that applies
`SimpleImputer(strategy="median")` + `StandardScaler` to the numeric columns and `OneHotEncoder(handle_unknown="ignore")` to
`"region"`, put it in a pipeline with `LogisticRegression(max_iter=2000)` and store the 5-fold CV accuracy in `ct_acc`. Store the
number of output features of the transformer in `n_out`.

In [ ]:
wine = load_wine(as_frame=True)
df_w = wine.data.copy()
r_w = np.random.default_rng(2)
df_w = df_w.mask(r_w.random(df_w.shape) < 0.05)        # 5% missing values
df_w["region"] = r_w.choice(["north", "south", "east"], size=len(df_w))
y_w = wine.target
num_cols = list(wine.data.columns); cat_cols = ["region"]
ct_acc = None
n_out = None

check('missing values present', bool(df_w[num_cols].isna().any().any()), True)
check('output features = 13 numeric + 3 one-hot', n_out, len(num_cols) + df_w['region'].nunique())
check('CV accuracy > 0.9', None if ct_acc is None else ct_acc > 0.9, True)

<details>
<summary><b>💡 Hint</b></summary>

`ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols), ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])`; `n_out = ct.fit_transform(df_w).shape[1]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Each column group gets its own preprocessing, and all of it is fitted inside each CV fold. `handle_unknown='ignore'` encodes unseen categories at prediction time as all zeros instead of crashing. Here `region` is pure noise, so the accuracy comes from the numeric features (about 0.97).

```python
wine = load_wine(as_frame=True)
df_w = wine.data.copy()
r_w = np.random.default_rng(2)
df_w = df_w.mask(r_w.random(df_w.shape) < 0.05)
df_w["region"] = r_w.choice(["north", "south", "east"], size=len(df_w))
y_w = wine.target
num_cols = list(wine.data.columns); cat_cols = ["region"]
ct = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
pipe_ct = make_pipeline(ct, LogisticRegression(max_iter=2000))
ct_acc = cross_val_score(pipe_ct, df_w, y_w, cv=cv5).mean()
n_out = clone(ct).fit_transform(df_w).shape[1]
print(ct_acc, n_out)
```

</details>

### Exercise 13 · A custom transformer
*💻 Code · ★★☆*

Write `QuantileClipper(BaseEstimator, TransformerMixin)` with hyperparameters `lower=0.01, upper=0.99`. `fit` learns the per-column
quantiles from the training data (store them as `lo_`, `hi_`, trailing underscore = learned), and `transform` clips to them.
Follow sklearn's conventions: `__init__` only stores the arguments, `fit` returns `self`. Check that it works inside a pipeline,
with `clone`, and that `get_params()` works.

In [ ]:
class QuantileClipper(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper
    def fit(self, X, y=None):
        # TODO: learn self.lo_ and self.hi_
        return self
    def transform(self, X):
        # TODO
        return None

qc = QuantileClipper(0.05, 0.95)
X_clipped = qc.fit(X_tr).transform(X_te)

check('clipping with training quantiles', X_clipped, np.clip(X_te, np.quantile(X_tr, 0.05, axis=0), np.quantile(X_tr, 0.95, axis=0)))
check('get_params', clone(qc).get_params(), {'lower': 0.05, 'upper': 0.95})

<details>
<summary><b>💡 Hint</b></summary>

`self.lo_ = np.quantile(X, self.lower, axis=0)`; `transform` returns `np.clip(X, self.lo_, self.hi_)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

`BaseEstimator` provides `get_params`/`set_params` by introspecting `__init__`'s arguments (which is why `__init__` must not transform them), and that is what `clone` and `GridSearchCV` (`quantileclipper__upper`) rely on. `TransformerMixin` adds `fit_transform`. For stateless functions, `FunctionTransformer(np.log1p)` is enough.

```python
class QuantileClipper(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper
    def fit(self, X, y=None):
        X = np.asarray(X, float)
        self.lo_ = np.quantile(X, self.lower, axis=0)
        self.hi_ = np.quantile(X, self.upper, axis=0)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, float), self.lo_, self.hi_)

qc = QuantileClipper(0.05, 0.95)
X_clipped = qc.fit(X_tr).transform(X_te)
print(cross_val_score(make_pipeline(QuantileClipper(), StandardScaler(), LogisticRegression(max_iter=1000)), X_tr, y_tr, cv=cv5).mean())
```

</details>

### Exercise 14 · A custom classifier
*💻 Code · ★★★*

Implement `CentroidClassifier(BaseEstimator, ClassifierMixin)`: `fit` stores `classes_` and the per-class mean vectors
`centroids_`; `predict` returns the class of the nearest centroid (Euclidean). Compare its predictions with
`sklearn.neighbors.NearestCentroid` on the scaled breast-cancer data, and show that it works with `cross_val_score` and
`GridSearchCV` (inside a pipeline with `StandardScaler`).

In [ ]:
class CentroidClassifier(BaseEstimator, ClassifierMixin):
    def fit(self, X, y):
        # TODO: self.classes_, self.centroids_
        return self
    def predict(self, X):
        return None

sc_bc = StandardScaler().fit(X_tr)
pred_cc = CentroidClassifier().fit(sc_bc.transform(X_tr), y_tr).predict(sc_bc.transform(X_te))
cv_cc = None   # mean CV accuracy of make_pipeline(StandardScaler(), CentroidClassifier())

check('same predictions as NearestCentroid', pred_cc, NearestCentroid().fit(sc_bc.transform(X_tr), y_tr).predict(sc_bc.transform(X_te)))
check('CV works and beats 0.9', None if cv_cc is None else cv_cc > 0.9, True)

<details>
<summary><b>💡 Hint</b></summary>

`self.classes_ = np.unique(y)`; `self.centroids_ = np.stack([X[y == c].mean(0) for c in self.classes_])`; predict with `argmin` over squared distances, then map back through `self.classes_`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

`ClassifierMixin` gives you `score` (accuracy) and tells CV utilities to use stratified folds. Returning `self.classes_[idx]` (not `idx`) keeps arbitrary labels working. Nearest centroid reaches about 92% CV accuracy here, a strong baseline in about ten lines.

```python
class CentroidClassifier(BaseEstimator, ClassifierMixin):
    def fit(self, X, y):
        X = np.asarray(X, float); y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.centroids_ = np.stack([X[y == c].mean(0) for c in self.classes_])
        return self
    def predict(self, X):
        X = np.asarray(X, float)
        d2 = ((X[:, None, :] - self.centroids_[None]) ** 2).sum(-1)
        return self.classes_[d2.argmin(1)]

sc_bc = StandardScaler().fit(X_tr)
pred_cc = CentroidClassifier().fit(sc_bc.transform(X_tr), y_tr).predict(sc_bc.transform(X_te))
cv_cc = cross_val_score(make_pipeline(StandardScaler(), CentroidClassifier()), X_tr, y_tr, cv=cv5).mean()
print(cv_cc)
```

</details>

### Exercise 15 · Comparing models on the same folds
*💻 Code · ★★☆*

Compare four models from the note with **the same** `cv5` splits on `X_tr, y_tr`: logistic regression and RBF-SVC (both with
`StandardScaler` in a pipeline), `RandomForestClassifier(n_estimators=200, random_state=0)` and
`GradientBoostingClassifier(random_state=0)`. Store a dict `{name: (mean, std)}` in `model_scores`. Why do tree ensembles not
need the scaler?

In [ ]:
model_scores = None

check('four models compared', None if model_scores is None else len(model_scores), 4)
check('all models above 0.9 CV accuracy', None if model_scores is None else all(m > 0.9 for m, s in model_scores.values()), True)

<details>
<summary><b>💡 Hint</b></summary>

Loop over a dict of estimators and call `cross_val_score(est, X_tr, y_tr, cv=cv5)` for each.

</details>

<details>
<summary><b>✅ Solution</b></summary>

All four land around 0.95–0.98, and the differences are within about one standard deviation across folds, so the data cannot really distinguish them; prefer the simpler model or use a paired test on the same folds. Trees split on thresholds of single features, so monotone rescaling does not change them. See [[Ensemble Methods]] and [[Support Vector Machines]].

```python
models = {
    'logreg': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'svc': make_pipeline(StandardScaler(), SVC(kernel='rbf')),
    'rf': RandomForestClassifier(n_estimators=200, random_state=0),
    'gb': GradientBoostingClassifier(random_state=0),
}
model_scores = {}
for name, est in models.items():
    s = cross_val_score(est, X_tr, y_tr, cv=cv5)
    model_scores[name] = (s.mean(), s.std())
    print(f'{name:7s} {s.mean():.3f} ± {s.std():.3f}')
```

</details>

### Exercise 16 · Choosing a decision threshold
*💻 Code · ★★★*

In the breast-cancer data, label `0` is **malignant**. Missing a malignant tumour is much worse than a false alarm, so we want
recall $\ge 0.98$ for the malignant class. Using `cross_val_predict(..., method="predict_proba")` with the logistic pipeline on
`X_tr, y_tr`, compute out-of-fold probabilities of malignancy, and find the **largest** threshold `thr` such that predicting
"malignant if $p\ge$ thr" gives recall $\ge0.98$ on those out-of-fold predictions. Store `thr` and the resulting precision `prec_thr`.

In [ ]:
thr = None
prec_thr = None

_p = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X_tr, y_tr, cv=cv5, method='predict_proba')[:, 0]
_m = (y_tr == 0)
if thr is None:
    check('recall >= 0.98 at thr', None, True)
else:
    check('recall >= 0.98 at thr', recall_score(_m, _p >= thr) >= 0.98, True)
    check('threshold is the largest such one', recall_score(_m, _p > thr) < 0.98 or thr >= _p[_m].max(), True)
    check('precision at thr', prec_thr, precision_score(_m, _p >= thr))

<details>
<summary><b>💡 Hint 1</b></summary>

Candidate thresholds are the out-of-fold probabilities of the malignant samples. Sort them in decreasing order.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Recall at threshold $t$ is the fraction of malignant samples with $p\ge t$; you need at least $\lceil0.98\,n_{mal}\rceil$ of them above it.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The default threshold 0.5 optimises accuracy-like behaviour, not your costs. Lowering the threshold trades precision for recall; choosing it on **out-of-fold** predictions (not on the test set) keeps the test estimate honest. `precision_recall_curve` gives the whole trade-off at once.

```python
p_mal = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X_tr, y_tr, cv=cv5, method='predict_proba')[:, 0]
is_mal = (y_tr == 0)
need = int(np.ceil(0.98 * is_mal.sum()))
thr = np.sort(p_mal[is_mal])[::-1][need - 1]        # the need-th largest malignant probability
prec_thr = precision_score(is_mal, p_mal >= thr)
print(thr, recall_score(is_mal, p_mal >= thr), prec_thr)
```

</details>

### Exercise 17 · The unsupervised recipe
*💻 Code · ★★☆*

Run the note's unsupervised recipe on iris: standardise, `PCA(n_components=2)`, then `KMeans(n_clusters=3, n_init=10,
random_state=0)` on the 2-D scores. Store the PCA's `explained_variance_ratio_` in `evr` and the adjusted Rand index between the
clusters and the true species in `ari`. Verify `evr` independently: it equals the top eigenvalues of the **correlation** matrix
divided by the number of features. Why?

In [ ]:
X_iris, y_iris = load_iris(return_X_y=True)
evr = None
ari = None

_ev = np.sort(np.linalg.eigvalsh(np.corrcoef(X_iris, rowvar=False)))[::-1]
check('explained variance ratio', evr, _ev[:2] / 4)
check('clusters agree with species (ARI > 0.5)', None if ari is None else ari > 0.5, True)

<details>
<summary><b>💡 Hint</b></summary>

After standardisation the covariance matrix equals the correlation matrix (up to the $N/(N-1)$ factor, which cancels in the ratio), whose trace is the number of features.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Standardised features have unit variance, so total variance $=d=4$ and the covariance is the correlation matrix. The first two PCs keep about 96% of the variance. k-means finds setosa perfectly but mixes versicolor and virginica a little (ARI about 0.6); clustering is not classification, as labels play no role. See [[PCA]] and [[Clustering]].

```python
X_iris, y_iris = load_iris(return_X_y=True)
pca = PCA(n_components=2)
Z = pca.fit_transform(StandardScaler().fit_transform(X_iris))
evr = pca.explained_variance_ratio_
labels = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(Z)
ari = adjusted_rand_score(y_iris, labels)
print(evr, ari)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [scikit-learn Recipes](scikit-learn%20Recipes.md).*